In [1]:
import uuid 
from typing import Dict, Any, Optional 
 
from google.adk.agents import Agent 
from google.adk.runners import InMemoryRunner 
from google.adk.tools import FunctionTool 
from google.genai import types 
from google.adk.events import Event 

In [2]:
# --- Define Tool Functions --- 
# These functions simulate the actions of the specialist agents. 
 
def booking_handler(request: str) -> str: 
   """ 
   Handles booking requests for flights and hotels. 
   Args: 
       request: The user's request for a booking. 
   Returns: 
       A confirmation message that the booking was handled. 
   """ 
   print("-------------------------- Booking Handler Called ----------------------------") 
   return f"Booking action for '{request}' has been simulated."

In [3]:
def info_handler(request: str) -> str: 
   """ 
   Handles general information requests. 
   Args: 
       request: The user's question. 
   Returns: 
       A message indicating the information request was handled. 
   """ 
   print("-------------------------- Info Handler Called ----------------------------") 
   return f"Information request for '{request}'. Result: Simulated information retrieval."

In [4]:
def unclear_handler(request: str) -> str: 
   """Handles requests that couldn't be delegated.""" 
   return f"Coordinator could not delegate request: '{request}'. Please clarify."

In [5]:
# --- Create Tools from Functions --- 
booking_tool = FunctionTool(booking_handler) 
info_tool = FunctionTool(info_handler) 

In [6]:
# Define specialized sub-agents equipped with their respective tools 
booking_agent = Agent( 
   name="Booker", 
   model="gemini-2.0-flash", 
   description="A specialized agent that handles all flight  and hotel booking requests by calling the booking tool.", 
   tools=[booking_tool] 
) 

In [7]:
info_agent = Agent( 
   name="Info", 
   model="gemini-2.0-flash", 
   description="A specialized agent that provides general information and answers user questions by calling the info tool.", 
   tools=[info_tool] 
) 

In [8]:
#Define the parents agent with explict delegation instructions

coordinator=Agent(
    name='Coordinator',
    model='gemini-2.0-flash',
    instruction=(
        'you are the main coordinator. Your only task is to analyse incoming user requests'
        'and delegate them to the appropriate specialist agent. Do not try to answer the user directly.'
        'for any requests related to booking flights or hotels, delegates to the "Booker" agent'
        'for all other general information questions, delegates to the "Info" agent.'
    ),
    description="A Coordinator that routes user requests to the correct specialist agent.",
    #the presence of sub_agents enables LLM-driven delegation (Auto-Flow) by default
    sub_agents=[booking_agent,info_agent]
)

In [9]:
# Execution Logic
# --- Execution Logic ---

async def run_coordinator(runner: InMemoryRunner, request: str):
    """
    Runs the coordinator agent with a given request and delegates.
    """
    print(f"\n--- Running Coordinator with request: '{request}' ---")
    final_result = ""

    try:
        user_id = "user_123"
        session_id = str(uuid.uuid4())

        # Create a new session
        await runner.session_service.create_session(
            app_name=runner.app_name,
            user_id=user_id,
            session_id=session_id
        )

        # Run the coordinator agent
        for event in runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=request)]
            ),
        ):

            # Check if this is the final response
            if event.is_final_response() and event.content:

                # Try to access text directly
                if hasattr(event.content, "text") and event.content.text:
                    final_result = event.content.text

                # Otherwise, combine text from all parts
                elif event.content.parts:
                    text_parts = [
                        part.text
                        for part in event.content.parts
                        if part.text
                    ]
                    final_result = "".join(text_parts)

                # Stop after receiving the final response
                break

        print(f"Coordinator Final Response: {final_result}")
        return final_result

    except Exception as e:
        print(f"An error occurred while processing your request: {e}")
        return f"An error occurred while processing your request: {e}"

In [10]:
async def main():
    """
    Main function to run the ADK example.
    """
    print("--- Google ADK Routing Example (ADK Auto-Flow Style) ---")
    print("Note: This requires Google ADK installed and authenticated.")

    # Create a runner using the coordinator agent
    runner = InMemoryRunner(coordinator)

    # Example 1: Booking request
    result_a = await run_coordinator(
        runner,
        "Book me a hotel in Paris."
    )
    print(f"Final Output A: {result_a}")

    # Example 2: General information question
    result_b = await run_coordinator(
        runner,
        "What is the highest mountain in the world?"
    )
    print(f"Final Output B: {result_b}")

    # Example 3: Random fact request
    result_c = await run_coordinator(
        runner,
        "Tell me a random fact."
    )  # Should go to Info
    print(f"Final Output C: {result_c}")

    # Example 4: Flight booking request
    result_d = await run_coordinator(
        runner,
        "Find flights to Tokyo next month."
    )  # Should go to Booker
    print(f"Final Output D: {result_d}")


if __name__ == "__main__":
    import nest_asyncio

    # Allow nested event loops (useful in Jupyter notebooks)
    nest_asyncio.apply()

    # Start the application
    await main()

c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(
Node execution failed with exception
Traceback (most recent call last):
  File "c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\workflow\_node_runner.py", line 129, in run
    await self._execute_node(ctx, node_input)
  File "c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\workflow\_node_runner.py", line 248, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\workflow\_node_runner.py", line 262, in _run_node_loop
    async for event in agen:
  File "c:\Users\Pappu Yadav\Desktop\A

--- Google ADK Routing Example (ADK Auto-Flow Style) ---
Note: This requires Google ADK installed and authenticated.

--- Running Coordinator with request: 'Book me a hotel in Paris.' ---


Root node Coordinator failed.
Traceback (most recent call last):
  File "c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\runners.py", line 800, in _cleanup_root_task
    await task
  File "C:\Users\Pappu Yadav\AppData\Local\Programs\Python\Python311\Lib\asyncio\futures.py", line 290, in __await__
    return self.result()  # May raise too.
           ^^^^^^^^^^^^^
  File "C:\Users\Pappu Yadav\AppData\Local\Programs\Python\Python311\Lib\asyncio\futures.py", line 203, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "C:\Users\Pappu Yadav\AppData\Local\Programs\Python\Python311\Lib\asyncio\tasks.py", line 277, in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\Users\Pappu Yadav\Desktop\Agentic-Design_pattern\Agentic_design_patterns\my_env\Lib\site-packages\google\adk\runners.py", line 568, in _drive_root_node
    raise ctx.error
  File "c:\Users\Pappu Yadav\Desktop\Agent

Coordinator Final Response: 
Final Output A: 

--- Running Coordinator with request: 'What is the highest mountain in the world?' ---
Coordinator Final Response: 
Final Output B: 

--- Running Coordinator with request: 'Tell me a random fact.' ---
Coordinator Final Response: 
Final Output C: 

--- Running Coordinator with request: 'Find flights to Tokyo next month.' ---
Coordinator Final Response: 
Final Output D: 
